# SingBERT Inference — NS Sentiment (Stage 5a — Full Corpus)

Runs SingBERT v7 fine-tuned model over all 737k NS Reddit chunks.

## What this produces
- `chunk_sentiment_singbert.parquet` — `chunk_id`, `sent_neg`, `sent_neu`, `sent_pos`
  (softmax probabilities, same schema as the old XLM `chunk_sentiment.parquet`)

## Kaggle datasets required
- `ns-sentiment-chunks-v3` — contains `comments_chunks.parquet` + `submissions_chunks.parquet`
- Fine-tuned model uploaded as a separate Kaggle dataset (upload `results-17/singbert_ns_sentiment/best_model/`)

## Runtime
~2–3 hrs on Kaggle T4 x2 GPU at batch_size=128

In [ ]:
!pip install -q -U transformers

In [ ]:
import glob
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ── Auto-discover input files ─────────────────────────────────────────────
def find_input_file(filename: str) -> Path:
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if not matches:
        raise FileNotFoundError(
            f"{filename} not found under /kaggle/input/\n"
            f"Attach the dataset containing {filename} via Add Data."
        )
    return Path(matches[0])

def find_input_dir(dirname: str) -> Path:
    matches = glob.glob(f"/kaggle/input/**/{dirname}", recursive=True)
    if not matches:
        raise FileNotFoundError(f"Directory '{dirname}' not found under /kaggle/input/")
    return Path(matches[0])

OUTPUT_DIR = Path("/kaggle/working")

# ── Config ────────────────────────────────────────────────────────────────
MAX_LEN    = 256
BATCH_SIZE = 128   # T4 x2 comfortably fits 128 for inference
LABEL2ID   = {"negative": 0, "neutral": 1, "positive": 2}
ID2LABEL   = {v: k for k, v in LABEL2ID.items()}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    n_gpu = torch.cuda.device_count()
    print(f"N GPUs : {n_gpu}")

In [ ]:
# ── Load model + tokenizer ────────────────────────────────────────────────
MODEL_DIR = find_input_dir("best_model")
print(f"Loading model from: {MODEL_DIR}")

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR))

# DataParallel for T4 x2
if torch.cuda.device_count() > 1:
    print(f"Using DataParallel across {torch.cuda.device_count()} GPUs")
    model = torch.nn.DataParallel(model)

model = model.to(DEVICE)
model.eval()
print("Model ready.")

In [ ]:
# ── Load all chunks ───────────────────────────────────────────────────────
comments_path     = find_input_file("comments_chunks.parquet")
submissions_path  = find_input_file("submissions_chunks.parquet")

comments    = pd.read_parquet(comments_path,    columns=["chunk_id", "text"])
submissions = pd.read_parquet(submissions_path, columns=["chunk_id", "text"])

chunks = pd.concat([comments, submissions], ignore_index=True)
chunks = chunks.drop_duplicates(subset="chunk_id").reset_index(drop=True)
chunks["text"] = chunks["text"].fillna("").astype(str)

print(f"Total unique chunks: {len(chunks):,}")
print(f"  comments   : {len(comments):,}")
print(f"  submissions: {len(submissions):,}")

In [ ]:
# ── Dataset + inference loop ──────────────────────────────────────────────
class ChunkDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            max_length=max_len,
            padding="max_length",
            return_tensors="pt",
        )

    def __len__(self):
        return self.encodings["input_ids"].shape[0]

    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.encodings.items()}


def run_inference(texts, model, tokenizer, batch_size, device):
    """Returns softmax probabilities [N, 3] (neg, neu, pos)."""
    dataset = ChunkDataset(texts, tokenizer, MAX_LEN)
    loader  = DataLoader(dataset, batch_size=batch_size,
                         shuffle=False, num_workers=2, pin_memory=True)

    all_probs = []
    total = len(loader)

    with torch.no_grad():
        for i, batch in enumerate(loader):
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(**batch).logits
            # Handle DataParallel output
            if isinstance(logits, torch.Tensor) and logits.dim() == 2:
                probs = torch.softmax(logits, dim=-1).cpu().numpy()
            all_probs.append(probs)

            if (i + 1) % 100 == 0 or (i + 1) == total:
                done = (i + 1) * batch_size
                print(f"  {done:>8,} / {len(texts):,}  ({(i+1)/total*100:.1f}%)",
                      flush=True)

    return np.vstack(all_probs)


print(f"Running inference on {len(chunks):,} chunks (batch_size={BATCH_SIZE})...")
probs = run_inference(
    chunks["text"].tolist(), model, tokenizer, BATCH_SIZE, DEVICE
)
print(f"Done. Output shape: {probs.shape}")

In [ ]:
# ── Build output parquet ──────────────────────────────────────────────────
out = pd.DataFrame({
    "chunk_id" : chunks["chunk_id"],
    "sent_neg"  : probs[:, 0].astype("float32"),
    "sent_neu"  : probs[:, 1].astype("float32"),
    "sent_pos"  : probs[:, 2].astype("float32"),
})

# Sanity checks
assert len(out) == len(chunks), "Row count mismatch"
assert out["chunk_id"].nunique() == len(out), "Duplicate chunk_ids"
prob_sums = (out["sent_neg"] + out["sent_neu"] + out["sent_pos"])
assert prob_sums.between(0.999, 1.001).all(), "Softmax doesn't sum to 1"

out_path = OUTPUT_DIR / "chunk_sentiment_singbert.parquet"
out.to_parquet(out_path, index=False)

print(f"Saved → {out_path}")
print(f"Rows  : {len(out):,}")
print(f"\nMajority label distribution:")
majority = out[["sent_neg","sent_neu","sent_pos"]].idxmax(axis=1).str.replace("sent_","")
print(majority.value_counts())
print(f"\nMean scores:")
print(f"  sent_neg : {out['sent_neg'].mean():.4f}")
print(f"  sent_neu : {out['sent_neu'].mean():.4f}")
print(f"  sent_pos : {out['sent_pos'].mean():.4f}")
print(f"\nSoftmax sum check (should be ~1.000):")
print(f"  mean={prob_sums.mean():.6f}  min={prob_sums.min():.6f}  max={prob_sums.max():.6f}")